In [0]:
%run ../../config/config

## Checks - Zerobus transport

Run after `05_producer_zerobus` and the Lakeflow pipeline `team-crypto-ticks-streaming`. Every check raises on failure, so the notebook can also run as a task in a Lakeflow Job

### Zerobus records moved from the landing table to bronze and silver

In [0]:
landed = spark.table(BRONZE_ZEROBUS_TICKS).count()
in_bronze = spark.table(BRONZE_EVENTHUB_TICKS).filter("source = 'zerobus'").count()
in_silver = spark.table(SILVER_TICKS).filter("transport_source = 'zerobus'").count()
in_quarantine = spark.table(SILVER_QUARANTINE).filter("source = 'zerobus'").count()

assert landed > 0, "No Zerobus records landed; run 05_producer_zerobus first"
# the append flow copies each landed row once: bronze can lag until the next pipeline run, never exceed
assert 0 < in_bronze <= landed, f"Bronze has {in_bronze} Zerobus rows, landing table has {landed}"
assert in_silver > 0, "No Zerobus ticks in silver"
# silver drops duplicates, quarantine keeps invalid rows, so together they never exceed bronze
assert in_silver + in_quarantine <= in_bronze, "Silver + quarantine exceed bronze for Zerobus rows"

### No lost records: producer_seq has no gaps in any producer run

In [0]:
gaps = spark.sql(f"""
    SELECT producer_id
    FROM {BRONZE_ZEROBUS_TICKS}
    GROUP BY producer_id
    HAVING count(DISTINCT producer_seq) <> max(producer_seq)
""").count()
assert gaps == 0, f"{gaps} producer runs have gaps in producer_seq, records were lost"